In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm

from model.dataset import BagsDataset, custom_collate_fn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
adata = sc.read_h5ad('../test.h5ad')

In [4]:
dataset = BagsDataset(adata, radius= 200, immune_cell='tcell')

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 5870.20it/s]

Total bags created: 3858
Average instances per bag: 8


In [5]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)


In [6]:
distances, gene_expressions, labels, core_index = next(iter(dataloader))

In [7]:
class Sparsemax(nn.Module):
    def __init__(self, dim=None):
        super(Sparsemax, self).__init__()
        self.dim = -1 if dim is None else dim

    def forward(self, input):
        input = input.transpose(0, self.dim)
        original_size = input.size()
        input = input.reshape(input.size(0), -1)
        input = input.transpose(0, 1)
        dim = 1

        number_of_logits = input.size(dim)

        input = input - torch.max(input, dim=dim, keepdim=True)[0].expand_as(input)

        zs = torch.sort(input=input, dim=dim, descending=True)[0]
        range = torch.arange(start=1, end=number_of_logits + 1, step=1, device=input.device, dtype=input.dtype).view(1, -1)
        range = range.expand_as(zs)

        bound = 1 + range * zs
        cumulative_sum_zs = torch.cumsum(zs, dim)
        is_gt = torch.gt(bound, cumulative_sum_zs).type(input.type())
        k = torch.max(is_gt * range, dim, keepdim=True)[0]

        zs_sparse = is_gt * zs

        taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
        taus = taus.expand_as(input)

        output = F.relu(input - taus)

        output = output.transpose(0, 1)
        output = output.reshape(original_size)
        output = output.transpose(0, self.dim)

        return output

In [8]:
model = Sparsemax()
output = model(gene_expressions[0])
output

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [9]:
print(torch.count_nonzero(gene_expressions[0] == 0)) 
print(torch.count_nonzero(output == 0)) 

tensor(1329)
tensor(3992)


In [10]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        self.sparsemax = Sparsemax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = torch.sigmoid(self.a)  
        x = self.sparsemax(-a * x)
        return x

In [11]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[0.2500],
        [0.2500],
        [0.2500],
        [0.2500],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000]], grad_fn=<TransposeBackward0>)


In [14]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        b = torch.sigmoid(self.b)
        x = b * x
        x = self.softmax(x)
        return x


In [15]:
gene_expressions[0].shape

torch.Size([8, 500])

In [16]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output.shape)

torch.Size([8, 500])


In [85]:
class Immunogenicity(nn.Module):
    def __init__(self):
        super(Immunogenicity, self).__init__()
        #self.ig = nn.Parameter(torch.rand(g) * (10 - 1) - 10, requires_grad=True)
        self.ig = nn.Parameter(torch.full((500,), -4.0),requires_grad=True)
    
    def forward(self):
        #sigmod 

        ig = torch.sigmoid(self.ig)
        return ig

In [103]:
class Immunogenicity(nn.Module):
    def __init__(self, num_genes, gene_name_to_index):
        super(Immunogenicity, self).__init__()
        self.num_genes = num_genes
        self.gene_name_to_index = gene_name_to_index
        self.ig = nn.Parameter(torch.full((self.num_genes,), -4.0), requires_grad=True)

    def forward(self, input_gene_indices):
        
        self.ig.data[input_gene_indices] = self.ig.data[input_gene_indices]
        ig = torch.sigmoid(self.ig)
        return ig

In [104]:
gene_df = pd.read_csv('human.csv')
gene_names = gene_df['Gene'].tolist()
gene_name_to_index = {name: i for i, name in enumerate(gene_names)}

In [105]:
model = Immunogenicity(len(gene_names), gene_name_to_index)


In [110]:
input_gene_names = adata.var_names
input_gene_names = [gene for gene in input_gene_names if gene in gene_name_to_index]
len(input_gene_names)
input_gene_indices = torch.tensor([gene_name_to_index[gene] for gene in input_gene_names], dtype=torch.long)


In [113]:
output = model(input_gene_indices)
print(output.shape)

torch.Size([27200])


In [86]:
model = Immunogenicity()

output = model()
output

tensor([0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180, 0.0180,
        0.0180, 0.0180, 0.0180, 0.0180, 

In [87]:
print(output.shape)

torch.Size([500])


In [88]:
class MIL(nn.Module):
    def __init__(self):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.Immunogenicity = Immunogenicity()
        #self.pooling = MILPooling()
    
    def forward(self, distances, gene_expressions):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            #print(distance.size())
            distance = self.distance(distance)
            #print("xxxxxxx")
            gene_expression = self.gene_expression(gene_expression)
            #print(gene_expression.shape)
            immunogenicity = self.Immunogenicity()
            #print(affinity,shape)
            #for j in range(len(gene_expression)):
                #zj = gene_expression[j, :] 
                #z.append(zj)
            z = gene_expression @ immunogenicity
            #print(z)
            #print("***")
            
            #print(z.shape)
            #print("***")
            #print(distance.squeeze().shape)
            #print(z)
            #print(z)
            z = z.unsqueeze(1)
            #print(z)
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            #print(bag_output)
            bag_output = torch.sigmoid(bag_output)
            bag_outputs.append(bag_output)
        return torch.stack(bag_outputs).squeeze(dim=1)
    




In [89]:
model = MIL()
output = model(distances, gene_expressions)

In [90]:
output

tensor([0.5045], grad_fn=<SqueezeBackward1>)

In [91]:
labels[0]


tensor(0.)

In [92]:
make_dot(output, params=dict(model.named_parameters())).render("MIL_computational_graph", format="png")


'MIL_computational_graph.png'

In [93]:
def model_summary(model, input_data):
    def register_hook(module):
        def hook(module, input, output):
            class_name = str(module.__class__).split(".")[-1].split("'")[0]
            module_idx = len(summary)

            m_key = "%s-%i" % (class_name, module_idx + 1)
            summary[m_key] = OrderedDict()
            
            if input:
                summary[m_key]["input_shape"] = list(input[0].size())
            else:
                summary[m_key]["input_shape"] = "No Input"
                
            try:
                summary[m_key]["output_shape"] = list(output.size())
            except AttributeError:
                summary[m_key]["output_shape"] = "No Output Size"
                
            summary[m_key]["trainable"] = any(p.requires_grad for p in module.parameters())
            summary[m_key]["nb_params"] = sum(p.numel() for p in module.parameters() if p.requires_grad)

        if not isinstance(module, nn.Sequential) and \
           not isinstance(module, nn.ModuleList) and \
           not (module == model):
            hooks.append(module.register_forward_hook(hook))

    # Create properties
    summary = OrderedDict()
    hooks = []

    # Register hook
    model.apply(register_hook)

    # Make a forward pass
    model(*input_data)

    # Remove these hooks
    for h in hooks:
        h.remove()

    # Print out the summary
    print("----------------------------------------------------------------")
    print("{:>20} {:>25} {:>15}".format("Layer (type)", "Output Shape", "Param #"))
    print("================================================================")
    total_params = 0
    total_output = 0
    trainable_params = 0
    for layer in summary:
        # Input shape, output shape, and number of parameters
        line_new = "{:>20} {:>25} {:>15}".format(
            layer,
            str(summary[layer]["output_shape"]),
            "{0:,}".format(summary[layer]["nb_params"]),
        )
        total_params += summary[layer]["nb_params"]
        if summary[layer]["output_shape"] != "Error" and summary[layer]["output_shape"] != "No Output Size":
            total_output += np.prod(summary[layer]["output_shape"])
        if "trainable" in summary[layer] and summary[layer]["trainable"]:
            trainable_params += summary[layer]["nb_params"]
        print(line_new)

    # Total params
    total_input_size = np.sum([np.prod(i.shape) for i in input_data if isinstance(i, torch.Tensor)]) * 4. / (1024 ** 2.)
    total_output_size = 2. * total_output * 4. / (1024 ** 2.)  # x2 for gradients
    total_params_size = total_params * 4. / (1024 ** 2.)
    total_size = total_params_size + total_output_size + total_input_size

    print("================================================================")
    print("Total params: {0:,}".format(total_params))
    print("Trainable params: {0:,}".format(trainable_params))
    print("Non-trainable params: {0:,}".format(total_params - trainable_params))
    print("----------------------------------------------------------------")
    print("Input size (MB): %0.2f" % total_input_size)
    print("Forward/backward pass size (MB): %0.2f" % total_output_size)
    print("Params size (MB): %0.2f" % total_params_size)
    print("Estimated Total Size (MB): %0.2f" % total_size)
    print("----------------------------------------------------------------")

In [94]:
model_summary(model, (distances, gene_expressions))


----------------------------------------------------------------
        Layer (type)              Output Shape         Param #
         Sparsemax-1                    [8, 1]               0
          Distance-2                    [8, 1]               1
           Softmax-3                  [8, 500]               0
   Gene_expression-4                  [8, 500]               1
    Immunogenicity-5                     [500]             500
Total params: 502
Trainable params: 502
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.06
Params size (MB): 0.00
Estimated Total Size (MB): 0.07
----------------------------------------------------------------


In [95]:
model = MIL()
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (softmax): Softmax(dim=-1)
  )
  (Immunogenicity): Immunogenicity()
)

In [96]:
num_epochs = 6

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()
            output = model(distances, gene_expressions)
            #print(output)
            #print(label)
            loss = criterion(output, label)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')


Epoch 1/6: 100%|██████████| 3858/3858 [00:08<00:00, 472.63batch/s, loss=0.684]


Epoch [1/6], Loss: 0.6920


Epoch 2/6:  11%|█         | 422/3858 [00:00<00:06, 517.71batch/s, loss=0.684]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

Epoch 2/6: 100%|██████████| 3858/3858 [00:07<00:00, 486.97batch/s, loss=0.684]


Epoch [2/6], Loss: 0.6920


Epoch 3/6:  10%|▉         | 368/3858 [00:00<00:06, 508.34batch/s, loss=0.684]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

Epoch 3/6: 100%|██████████| 3858/3858 [00:07<00:00, 484.90batch/s, loss=0.702]


Epoch [3/6], Loss: 0.6920


Epoch 4/6: 100%|██████████| 3858/3858 [00:08<00:00, 472.78batch/s, loss=0.702]


Epoch [4/6], Loss: 0.6920


Epoch 5/6:  98%|█████████▊| 3796/3858 [00:07<00:00, 518.78batch/s, loss=0.684]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

Epoch 6/6: 100%|██████████| 3858/3858 [00:07<00:00, 489.45batch/s, loss=0.702]

Epoch [6/6], Loss: 0.6919


In [64]:
def predict(model, adata, device, radius=200, max_instances=None):
    model.eval()
    
    dataset = BagsDataset(adata_radius_input=adata, immune_cell='tcell', radius=radius, max_instances=max_instances)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = np.full(len(adata.obs), np.nan)  # Initialize predictions array with NaNs

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, _, core_idx in tepoch:
                tepoch.set_description("Predicting")
                
                # Move data to the device
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                
                output = model(distances, gene_expressions)
                predictions[core_idx.item()] = output.cpu().numpy().flatten()

    adata.obs['tcr_predict'] = predictions
    return adata


In [65]:
adata = predict(model, adata,radius=200, device=device)

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 5939.36it/s]


Total bags created: 3858
Average instances per bag: 8


Predicting:   0%|          | 0/3858 [00:00<?, ?batch/s]/tmp/ipykernel_8840/3842026426.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions[core_idx.item()] = output.cpu().numpy().flatten()
Predicting:  75%|███████▌  | 2899/3858 [00:02<00:00, 1171.56batch/s]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [66]:
adata.obs

,X,Y,T,cell_type,tcr_predict
AACACGTGCATCGCAC-1,7600,2200,0,0,0.504476
AACACTTGGCAAGGAA-1,4700,7100,1,1,0.501798
AACAGGATTCATAGTT-1,4900,4300,1,0,0.501226
AACAGGTTATTGCACC-1,2800,8600,0,0,0.513621
AACAGGTTCACCGAAG-1,5100,4100,1,0,0.500040
...,...,...,...,...,...
TGTTGGAACCTTCCGC-1,3500,3500,0,0,0.513246
TGTTGGAACGAGGTCA-1,2800,7200,0,1,0.506220
TGTTGGAAGCTCGGTA-1,100,9500,0,0,0.501830
TGTTGGATGGACTTCT-1,1300,5300,0,0,0.502015
